In [1]:
import os 
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
import datetime

import sys
sys.path.append(r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\ccf_slice_registration')
from ccf_slice_registration_functions_withUpright import *
from soma_and_fiducial_pins_new import *

%matplotlib inline


Load CCF


In [2]:
# read in CCF (with fixed headers)
image_directory =  r'\\allen\programs\celltypes\workgroups\mousecelltypes\_UPENN_fMOST\mouse_ccf_fixed_headers_um\average_template'
ccf_file = os.path.join(image_directory, "average_template_10.nii.gz" )
resolution = 10
ccf = sitk.ReadImage( ccf_file )
volume_shape = ccf.GetSize()
z_size = volume_shape[2]*resolution
z_midline = z_size / 2

Get pins

In [3]:
pins_path = get_soma_and_fiducial_pins()
pins = pd.read_csv(pins_path)

pins['specimen_name'] = pins['specimen_name'].str.strip() #strip any erroneous white space from the start and end fo specimen names
pins = pins[pins['specimen_name'] != 'Point'] #remove one random pin on a cortical slice. 

#pins contains fiducial pins (specimen name ends in a letter) and soma pins (specimen name ends in a number)
#break up pins into fiducial vs soma pin dataframes
fiducials_dict = {}
somas_dict = {}
for i, p in pins.iterrows():
    last_pin_char = p['specimen_name'][-1]
    if last_pin_char.isalpha(): fiducials_dict[i] = p #the last char of the pin name is a letter, so this is a fiducial (not a soma) pin 
    else: somas_dict[i] = p #the last char of the pin name is a number, so this is a soma (not a fiducial) pin
        
somas = pd.DataFrame.from_dict(somas_dict, orient='index').reset_index().drop(['index'], axis=1)
fiducials = pd.DataFrame.from_dict(fiducials_dict, orient='index').reset_index().drop(['index'], axis=1)

slices = fiducials.slide_specimen_id.unique() #get the slices that have fiducials
print('{} slices with fiducials'.format(len(slices)))

{'slide_specimen_id': 1208614358, 'specimen_name': 'Curve'}
{'slide_specimen_id': 1266114320, 'specimen_name': 'Gad2-IRES-Cre;Ai14-670842.06.02.01'}
{'slide_specimen_id': 1374540601, 'specimen_name': 'C57BL6J-742760.03.02.01'}
{'slide_specimen_id': 1262448162, 'specimen_name': 'Sst-IRES-Cre;Ai14-670419.09.03.02'}
{'slide_specimen_id': 1396808298, 'specimen_name': 'Pvalb-IRES-Cre;Ai14-756291.06.04b'}
{'slide_specimen_id': 1396808298, 'specimen_name': 'Pvalb-IRES-Cre;Ai14-756291.06.04c'}
{'slide_specimen_id': 1243457003, 'specimen_name': 'Pvalb-IRES-Cre;Ai14-660824.06.02.01'}
382 slices with fiducials


In [8]:
pins[pins.slide_specimen_id.isin([645314706])].specimen_name.tolist()

['Slc17a8-IRES2-Cre;Slc32a1-IRES2-FlpO;Ai65-354751.03.01.01',
 'Slc17a8-IRES2-Cre;Slc32a1-IRES2-FlpO;Ai65-354751.03.01.02',
 'Slc17a8-IRES2-Cre;Slc32a1-IRES2-FlpO;Ai65-354751.03.01a',
 'Slc17a8-IRES2-Cre;Slc32a1-IRES2-FlpO;Ai65-354751.03.01b',
 'Slc17a8-IRES2-Cre;Slc32a1-IRES2-FlpO;Ai65-354751.03.01c']

Pulls swcs from a set of dirs, rather than from LIMS

In [4]:
#From swc dirs, get specimen ids to register
swc_root = r'\\allen\programs\celltypes\workgroups\mousecelltypes\Matt_Mallory\scripts\Feature_Calculation_Clean\Feature_Data\BasalGanglia_Sept_2025\SWC_Files\PSEQ_New'
swc_dirs = ['ForSWB_AB_QCd_AutotraceJuliaPassed_Vox', 'ForSWB_All_AutotraceJuliaPassed_Vox']

#root with already computed slice transforms
slice_dir = r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\ccf_slice_registration\ccf_reg_output_20251001'

#TOGGLE
slice_subset = [645314706]

for swc_dir in swc_dirs:
    in_dir = os.path.join(swc_root, swc_dir)
    out_dir = os.path.join(swc_root, swc_dir+'_ccf_reg')
    os.makedirs(out_dir, exist_ok=True)

    specimen_ids = [int(x.rsplit('.swc', 1)[0]) for x in os.listdir(in_dir)]
    if not len(specimen_ids) == len(set(specimen_ids)):
        print('Warning: specimen_ids are not unique!')

    registered_cells = []
    uprighted_cells = []
    cells_with_issues = {}

    for sp_id in specimen_ids:

        try:
            sp_name = get_name_by_id(sp_id)
            sl_name = sp_name.rsplit('.', 1)[0]
            sl_id = get_id_by_name(sl_name)

            if not sl_id in slice_subset:
                continue

            #check if there's a transform for this slice 
            slice_path = os.path.join(slice_dir, sl_name)
            if not os.path.isfile(os.path.join(slice_path, 'overview_to_virtual_slice_transform.txt')): continue #go on to next cell 
            if not os.path.isfile(os.path.join(slice_path, 'virtual_slice_to_ccf_transform.txt')): continue #go on to next cell 
            if not os.path.isfile(os.path.join(slice_path, 'alignment_output.csv')): continue #go on to next cell 


            #get soma loc in 20x 
            alignment_output = pd.read_csv(os.path.join(slice_path, 'alignment_output.csv'))
            this_cell_alignment = alignment_output.query("draw_type == 'Soma'")[alignment_output.specimen_name == sp_name]
            if len(this_cell_alignment) == 0: 
                cells_with_issues[sp_name] = 'No soma pin'
                print('\t no soma pin')
                continue
            if len(this_cell_alignment) > 1: 
                cells_with_issues[sp_name] = 'Multiple soma pins'
                print('\t multiple soma pins')
                continue
            lims_soma = this_cell_alignment['center_micron'].values
            lims_soma = lims_soma[0][1:-2].split(' ')
            lims_soma = [float(i) for i in lims_soma if len(i) > 0]

            #load affine transforms
            overview_to_virtual_slice_transform =           sitk.ReadTransform(os.path.join(slice_path, 'overview_to_virtual_slice_transform.txt'))
            overview_to_virtual_slice_upright_transform =   sitk.ReadTransform(os.path.join(slice_path, 'overview_to_virtual_slice_upright_transform.txt'))
            virtual_slice_to_ccf_transform =                sitk.ReadTransform(os.path.join(slice_path, 'virtual_slice_to_ccf_transform.txt'))


            #register the swc in the given dir 
            swc_path = os.path.join(in_dir, f'{sp_id}.swc')
            swc_name = swc_path.rsplit('\\',1)[1].split('.swc',1)[0]

            #register to CCF
            morph = to_dict(swc_path)
            morph = convert_pixel_to_um_dictnrn(morph, sp_id)
            register_morph(sp_name, sp_id, lims_soma, morph, out_dir, swc_name, somas, 
                    overview_to_virtual_slice_transform, virtual_slice_to_ccf_transform,
                    resolution, volume_shape, z_midline)
            registered_cells = registered_cells + [sp_name]

            #upright 
            morph = to_dict(swc_path)
            morph = convert_pixel_to_um_dictnrn(morph, sp_id)
            upright_morph(sp_name, sp_id, lims_soma, morph, out_dir, swc_name, somas, 
                        overview_to_virtual_slice_upright_transform, virtual_slice_to_ccf_transform,
                        resolution, volume_shape, z_midline)
            uprighted_cells = uprighted_cells + [sp_name]
        
        except: 
            cells_with_issues[sp_name] = 'Issue with this cell'
            continue

       
    cells_with_issues_df = pd.DataFrame.from_dict(list(cells_with_issues.items())) 
    cells_with_issues_df.to_csv(os.path.join(out_dir, 'cells_with_issues.csv'), index=False)

    registered_cells_df = pd.DataFrame(registered_cells)
    registered_cells_df.to_csv(os.path.join(out_dir, 'registered_cells.csv'), index=False)
        

C:\Users\sarah.wallingbell\AppData\Local\Temp\ipykernel_4884\2501960305.py:43: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  this_cell_alignment = alignment_output.query("draw_type == 'Soma'")[alignment_output.specimen_name == sp_name]
